In [1]:
from http.cookiejar import cut_port_re

import numpy as np
import pandas as pd
import re # 두우노큐식정?

# 내용 확인용

In [27]:
# 제조업만
nodong = pd.read_csv('data/노동비용(제조업).csv')
law_nodong = pd.read_csv('data/법정노동비용(제조업).csv')
notlaw_nodong = pd.read_csv('data/법정외_복지비용(제조업).csv')

# 전업종
nodong_total = pd.read_csv('data/노동비용(전업종).csv')
law_nodong_total = pd.read_csv('data/법정노동비용(전업종).csv')
notlaw_nodong_total = pd.read_csv('data/법정외_복지비용(전업종).csv')

# 산재
accident = pd.read_csv('data/전체재해현황및분석(제조업).csv')
accident_total = pd.read_csv('data/전체재해현황및분석(전업종).csv')

# 산업 규모뵬 임금, 근로시간
payment_time = pd.read_csv('data/산업규모및임금별근로시간(제조업).csv')
payment_time_total = pd.read_csv('data/산업규모및임금별근로시간(전업종).csv')
payment_time_part1 = pd.read_csv('data/payment_time_part1.csv')
payment_time_part2 = pd.read_csv('data/payment_time_part2.csv')

# 손익게산서 관련
sonic = pd.read_csv('data/손익계산서.csv')
sonic_jp = pd.read_csv('data/손익지표.csv')
sonic_jejo = pd.read_csv('data/손익계산서(제조업).csv')
sonic_jp_jejo = pd.read_csv('data/손익지표(제조업).csv')

In [ ]:
print(f'{nodong.shape} | {law_nodong.shape} | {notlaw_nodong.shape}')
print(f'{nodong_total.shape} | {law_nodong_total.shape} | {notlaw_nodong_total.shape}')
print(f'{accident.shape} | {accident_total.shape} | {payment_time.shape} | {payment_time_total.shape}')
print(f'{sonic.shape} | {sonic_jp.shape} | {sonic_jejo.shape} | {sonic_jp_jejo.shape}')

## nodong

## 헤더 나가!!

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
nodong.columns = nodong.iloc[0]
nodong_total.columns = nodong_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
nodong = nodong.drop(0)
nodong_total = nodong_total.drop(0)

# 너도 나가!
nodong.drop('기업규모별', axis=1, inplace=True)
nodong_total.drop('기업규모별', axis=1, inplace=True)

In [ ]:
nodong

In [ ]:
nodong_total

## 헤더 뭐뭐 있나 확인하기
- 근데... 뒤에서 가공해서 인스턴스로 다 빠질거면 굳이 단위는 안 빼도 되지 않나...?

In [ ]:
# 우리 아직 하나 더 남았습니다.
# 년도 앞에 2020~2024를 붙여야되는데 이게 10개단위거든요?
# 그죠 반복문 돌려야죠. 어느세월에 손으로 다 쓰고 앉아있음?

current_cols = nodong.columns.tolist() # 리스트로 가져옴
current_cols

# 여기서 0번 빼고 1번부터 바꿀거예요.
current_year = current_cols[1:]
print(current_year, len(current_year)) # 아 10개씩 끊으면 되네?

## 괄호씨는 더 이상 우리와 함께 할 수 없습니다.
- 아 아쉽습니다... 근데 빼야됩니다.

In [ ]:
pattern = r' \(천원\)' # 두유노정규식
current_cols = [re.sub(pattern, '', col) for col in current_cols]

current_cols

## 연도 앞으로 빼기

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['노동비용총액', '직접노동비용(계)', '정액 및 초과급여', '상여금 및 성과금', '간접노동비용(계)',
              '퇴직급여 등의 비용', '법정노동비용', '법정외 복지비용', '채용관련비용(모집비)', '교육훈련비용']

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(nodong.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
nodong.columns = final_cols
nodong_total.columns = final_cols # 둘이 범위만 다르고 칼럼이 같아서 이게 가능한겁니다.

In [ ]:
nodong_total

## 사르르르르르르

In [ ]:
nodong_melted = nodong.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')
nodong_total_melted = nodong_total.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')

nodong_melted[['연도', '항목']] = nodong_melted['항목'].str.split('_', expand=True, n=1)
nodong_total_melted[['연도', '항목']] = nodong_total_melted['항목'].str.split('_', expand=True, n=1)

nodong_melted['비용'] = pd.to_numeric(nodong_melted['비용'], errors='coerce')
nodong_total_melted['비용'] = pd.to_numeric(nodong_total_melted['비용'], errors='coerce')

nodong_melted = nodong_melted[['산업분류', '연도', '항목', '비용']]
nodong_total_melted = nodong_total_melted[['산업분류', '연도', '항목', '비용']]

In [ ]:
nodong_melted

In [ ]:
nodong_total_melted

## 저 앞에 알파벳들 되게 거슬린다 그죠?

In [ ]:
nodong_total_list = nodong_total_melted['산업분류'].tolist()
nodong_total_list

In [23]:
re_pattern = r'[A-Z]{1}.' # 규식정 출동
nodong_total_list = [re.sub(re_pattern, '', col) for col in nodong_total_list]

nodong_total_melted['산업분류'] = nodong_total_list

nodong_total_melted

NameError: name 'nodong_total_list' is not defined

## 저 숫자도 되게 거슬린다 그죠?
- (00~00) 이거요.

In [ ]:
no_braket = r'\([0-9]+~?[0-9]+\)'
nodong_total_list = [re.sub(no_braket, '', col) for col in nodong_total_list]

nodong_total_melted['산업분류'] = nodong_total_list

nodong_total_melted

## 돈단위 변경(천원->만원)

In [ ]:
nodong_melted['비용(만원)'] = nodong_melted['비용'] / 10
nodong_total_melted['비용(만원)'] = nodong_total_melted['비용'] / 10

In [ ]:
nodong_melted

## 저장_최종.csv

In [ ]:
nodong_melted.to_csv('data/nodong.csv', index=False) # 다음에는 후가공 필요없이 이걸로 하면 되지요.
nodong_total_melted.to_csv('data/nodong_total.csv', index=False)

# law_nodong

In [4]:
law_nodong

,기업규모별,산업분류,2019,2019.1,2019.2,2019.3,2019.4,2019.5,2019.6,2019.7,...,2024.4,2024.5,2024.6,2024.7,2024.8,2024.9,2024.10,2024.11,2024.12,2024.13
0,기업규모별,산업분류,법정노동비용(계) (천원),건강보험료 (천원),산재보험료 (천원),국민연금 (천원),고용보험료 (천원),장애인고용부담금 (천원),재해보상비 (천원),구성비(계) (%),...,고용보험료 (천원),장애인고용부담금 (천원),재해보상비 (천원),구성비(계) (%),건강보험료 (%),산재보험료 (%),국민연금 (%),고용보험료 (%),장애인고용부담금 (%),재해보상비 (%)
1,전규모(상용근로자 10인 이상),C.제조업(10~34),428.1,165.4,51.4,147.5,56.3,6.4,1.1,100,...,79.7,6.1,0.5,100,41.7,10,32.3,14.8,1.1,0.1
2,전규모(상용근로자 10인 이상),식료품 제조업,340.8,126,45.9,122.5,41.5,4.6,0.3,100,...,53.3,4.9,0.4,100,40.7,11.9,33.4,12.8,1.2,0.1
3,전규모(상용근로자 10인 이상),음료 제조업,483.2,179.9,59.9,163.4,70.1,7.7,2.2,100,...,93,6.5,0.5,100,40.1,10.6,32.8,15.3,1.1,0.1
4,전규모(상용근로자 10인 이상),담배 제조업,597.8,244.3,57.3,188.4,106.2,1.5,0,100,...,148.8,14,0,100,42.9,7.8,29.3,18.3,1.7,0
5,전규모(상용근로자 10인 이상),섬유제품 제조업; 의복 제외,284.9,109.8,37.5,103.9,29,4.7,0,100,...,34.5,1.8,0,100,44,10.4,34.9,10.3,0.5,0
6,전규모(상용근로자 10인 이상),"의복, 의복 액세서리 및 모피제품 제조업",328.5,120.1,32,124,40.7,11.7,0,100,...,48.4,7.5,0,100,41.3,8.3,35.9,12.6,2,0
7,전규모(상용근로자 10인 이상),"가죽, 가방 및 신발 제조업",308.9,123,34,109,35.1,5.4,2.5,100,...,38.6,1.8,0.3,100,45.5,10.6,31.7,11.7,0.5,0.1
8,전규모(상용근로자 10인 이상),목재 및 나무제품 제조업; 가구 제외,357.8,135.8,68.7,119.1,33.6,0.5,0,100,...,42,0.5,0.8,100,42.3,14.6,32.3,10.5,0.1,0.2
9,전규모(상용근로자 10인 이상),"펄프, 종이 및 종이제품 제조업",356.5,135.3,60.1,119.8,39.5,1.8,0,100,...,49.6,3,0,100,41.3,14.8,32.1,11.1,0.7,0


## 헤더 나가! 

In [5]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
law_nodong.columns = law_nodong.iloc[0]
law_nodong_total.columns = law_nodong_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
law_nodong = law_nodong.drop(0)
law_nodong_total = law_nodong_total.drop(0)

# 너도 나가!
law_nodong.drop('기업규모별', axis=1, inplace=True)
law_nodong_total.drop('기업규모별', axis=1, inplace=True)

In [ ]:
law_nodong

In [ ]:
law_nodong_total

In [6]:
current_cols = law_nodong_total.columns
current_cols

Index(['산업분류', '법정노동비용(계) (천원)', '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)',
       '고용보험료 (천원)', '장애인고용부담금 (천원)', '재해보상비 (천원)', '구성비(계) (%)', '건강보험료 (%)',
       '산재보험료 (%)', '국민연금 (%)', '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)',
       '법정노동비용(계) (천원)', '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)',
       '장애인고용부담금 (천원)', '재해보상비 (천원)', '구성비(계) (%)', '건강보험료 (%)', '산재보험료 (%)',
       '국민연금 (%)', '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)', '법정노동비용(계) (천원)',
       '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)', '장애인고용부담금 (천원)',
       '재해보상비 (천원)', '구성비(계) (%)', '건강보험료 (%)', '산재보험료 (%)', '국민연금 (%)',
       '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)', '법정노동비용(계) (천원)',
       '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)', '장애인고용부담금 (천원)',
       '재해보상비 (천원)', '구성비(계) (%)', '건강보험료 (%)', '산재보험료 (%)', '국민연금 (%)',
       '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)', '법정노동비용(계) (천원)',
       '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)', '장애인고용부담금 (천원)',
       '재해보상비 (천원)'

## 괄호 나가

In [7]:
pattern = r' \(천원\)' # 두유노정규식
current_cols = [re.sub(pattern, '', col) for col in current_cols] # (천원은 다 빼주시고)
current_cols = [re.sub(r' \(%\)', '_(%)', col) for col in current_cols] # 퍼센트도 퇴근합니당

current_cols

['산업분류',
 '법정노동비용(계)',
 '건강보험료',
 '산재보험료',
 '국민연금',
 '고용보험료',
 '장애인고용부담금',
 '재해보상비',
 '구성비(계)_(%)',
 '건강보험료_(%)',
 '산재보험료_(%)',
 '국민연금_(%)',
 '고용보험료_(%)',
 '장애인고용부담금_(%)',
 '재해보상비_(%)',
 '법정노동비용(계)',
 '건강보험료',
 '산재보험료',
 '국민연금',
 '고용보험료',
 '장애인고용부담금',
 '재해보상비',
 '구성비(계)_(%)',
 '건강보험료_(%)',
 '산재보험료_(%)',
 '국민연금_(%)',
 '고용보험료_(%)',
 '장애인고용부담금_(%)',
 '재해보상비_(%)',
 '법정노동비용(계)',
 '건강보험료',
 '산재보험료',
 '국민연금',
 '고용보험료',
 '장애인고용부담금',
 '재해보상비',
 '구성비(계)_(%)',
 '건강보험료_(%)',
 '산재보험료_(%)',
 '국민연금_(%)',
 '고용보험료_(%)',
 '장애인고용부담금_(%)',
 '재해보상비_(%)',
 '법정노동비용(계)',
 '건강보험료',
 '산재보험료',
 '국민연금',
 '고용보험료',
 '장애인고용부담금',
 '재해보상비',
 '구성비(계)_(%)',
 '건강보험료_(%)',
 '산재보험료_(%)',
 '국민연금_(%)',
 '고용보험료_(%)',
 '장애인고용부담금_(%)',
 '재해보상비_(%)',
 '법정노동비용(계)',
 '건강보험료',
 '산재보험료',
 '국민연금',
 '고용보험료',
 '장애인고용부담금',
 '재해보상비',
 '구성비(계)_(%)',
 '건강보험료_(%)',
 '산재보험료_(%)',
 '국민연금_(%)',
 '고용보험료_(%)',
 '장애인고용부담금_(%)',
 '재해보상비_(%)',
 '법정노동비용(계)',
 '건강보험료',
 '산재보험료',
 '국민연금',
 '고용보험료',
 '장애인고용부담금',
 '재해보상비',
 '구성비(계)_(%)',
 '건강보험료_(%)',
 

## 연도 나와

In [8]:
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['법정노동비용(계)','건강보험료','산재보험료','국민연금','고용보험료','장애인고용부담금','재해보상비','구성비(계)','건강보험료_(%)','산재보험료_(%)','국민연금_(%)','고용보험료_(%)','장애인고용부담금_(%)','재해보상비_(%)']

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(law_nodong.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
law_nodong.columns = final_cols
law_nodong_total.columns = final_cols

In [9]:
law_nodong_total

,산업분류,2019_법정노동비용(계),2019_건강보험료,2019_산재보험료,2019_국민연금,2019_고용보험료,2019_장애인고용부담금,2019_재해보상비,2019_구성비(계),2019_건강보험료_(%),...,2024_고용보험료,2024_장애인고용부담금,2024_재해보상비,2024_구성비(계),2024_건강보험료_(%),2024_산재보험료_(%),2024_국민연금_(%),2024_고용보험료_(%),2024_장애인고용부담금_(%),2024_재해보상비_(%)
1,전체,381.7,144.6,50.4,128.3,50.5,7.1,0.8,100,37.9,...,69.5,5.7,0.5,100,41.5,10.8,32,14.5,1.2,0.1
2,B.광업(05~08),448.5,139.9,151.7,111.2,37.8,0,7.9,100,31.2,...,51.6,0,2,100,37.1,31.9,21.1,9.5,0,0.4
3,C.제조업(10~34),428.1,165.4,51.4,147.5,56.3,6.4,1.1,100,38.6,...,79.7,6.1,0.5,100,41.7,10,32.3,14.8,1.1,0.1
4,"D.전기, 가스, 증기 및 공기 조절 공급업(35)",605.5,245.4,63.3,191.3,101.7,2.3,1.5,100,40.5,...,129.7,3,0,100,41.6,8.2,32,17.8,0.4,0
5,"E.수도, 하수 및 폐기물 처리, 원료 재생업(36~39)",315.6,122.7,47.8,109.4,33.2,1.5,1,100,38.9,...,44.5,0.8,0.1,100,46.8,9.8,32,11.2,0.2,0
6,F.건설업(41~42),560.3,145.4,188.5,131.3,71,20.8,3.4,100,26,...,78.3,6.1,2.4,100,31.2,29.7,25,12.7,1,0.4
7,G.도매 및 소매업(45~47),335.2,129,33.6,126,41.2,5.3,0.1,100,38.5,...,58.7,5.9,0,100,42.8,8.1,34.7,13.1,1.3,0
8,H.운수 및 창고업(49~52),307.8,119.1,39.5,106.7,37.4,4.6,0.5,100,38.7,...,60.2,5.8,0.6,100,41.3,11.4,31.5,14.3,1.4,0.1
9,I.숙박 및 음식점업(55~56),249.9,91.3,26.1,94,34,4.5,0,100,36.5,...,45.3,2.1,0,100,40.9,8.6,34.8,14.9,0.7,0
10,J.정보통신업(58~63),423.3,166.3,37.7,151.4,58.4,8.9,0.7,100,39.3,...,76.1,6,0,100,42.2,7.1,35,14.6,1.2,0


## 잠시만요 분리좀 하고 가실게요!
- 단위가 (천원)인 것과 (%)인 걸로 분리할겁니다.

In [17]:
# 공통분모
union_column = ['산업분류']

# 그룹 1: 단위가 (천원)임 ('법정노동비용(계) (천원)', '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)','장애인고용부담금 (천원)', '재해보상비 (천원)'
won_mnu_cols = [
    col for col in law_nodong.columns
    if any(x in col for x in ['법정노동비용(계)','건강보험료','산재보험료','국민연금','고용보험료','장애인고용부담금','재해보상비'])
    and '%' not in col  # 이 조건을 추가해서 (%) 항목을 걸러냅니다.
]
law_nodong_mnu_won = law_nodong_total.melt(id_vars=union_column, value_vars=won_mnu_cols, var_name='항목', value_name='비용')

# 그룹 2: % '구성비(계) (%)', '건강보험료 (%)', '산재보험료 (%)', '국민연금 (%)', '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)'
rate_mnu_cols = [col for col in law_nodong.columns if any(x in col for x in ['구성비(계)','건강보험료_(%)','산재보험료_(%)','국민연금_(%)','고용보험료_(%)','장애인고용부담금_(%)','재해보상비_(%)'])]
law_nodong_mnu_rate = law_nodong_total.melt(id_vars=union_column, value_vars=rate_mnu_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [law_nodong_mnu_won, law_nodong_mnu_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [18]:
# 공통분모
union_column = ['산업분류']

# 그룹 1: 단위가 (천원)임 ('법정노동비용(계) (천원)', '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)','장애인고용부담금 (천원)', '재해보상비 (천원)'
won_cols = [
    col for col in law_nodong.columns
    if any(x in col for x in ['법정노동비용(계)','건강보험료','산재보험료','국민연금','고용보험료','장애인고용부담금','재해보상비'])
    and '%' not in col  # 이 조건을 추가해서 (%) 항목을 걸러냅니다.
]
law_nodong_won = law_nodong_total.melt(id_vars=union_column, value_vars=won_cols, var_name='항목', value_name='비용')

# 그룹 2: % '구성비(계) (%)', '건강보험료 (%)', '산재보험료 (%)', '국민연금 (%)', '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)'
rate_cols = [col for col in law_nodong_total.columns if any(x in col for x in ['구성비(계)','건강보험료_(%)','산재보험료_(%)','국민연금_(%)','고용보험료_(%)','장애인고용부담금_(%)','재해보상비_(%)'])]
law_nodong_rate = law_nodong_total.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [law_nodong_won, law_nodong_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [19]:
law_nodong_won = law_nodong_won[['산업분류', '연도', '지표' ,'비용']]
law_nodong_mnu_won = law_nodong_mnu_won[['산업분류', '연도', '지표' ,'비용']]
law_nodong_won['지표'].value_counts()

지표
법정노동비용(계)    96
건강보험료        96
산재보험료        96
국민연금         96
고용보험료        96
장애인고용부담금     96
재해보상비        96
Name: count, dtype: int64

In [20]:
law_nodong_rate = law_nodong_rate[['산업분류', '연도', '지표' ,'수치']]
law_nodong_mnu_rate = law_nodong_mnu_rate[['산업분류', '연도', '지표' ,'수치']]
law_nodong_rate['지표'].value_counts()

지표
구성비(계)          96
건강보험료_(%)       96
산재보험료_(%)       96
국민연금_(%)        96
고용보험료_(%)       96
장애인고용부담금_(%)    96
재해보상비_(%)       96
Name: count, dtype: int64

## 너도 알파벳 떼고 나가자

In [21]:
law_nodong_won_list = law_nodong_won['산업분류'].tolist()
law_nodong_rate_list = law_nodong_rate['산업분류'].tolist()

In [24]:
re_pattern = r'[A-Z]{1}.' # 규식정 출동

law_nodong_rate_list = [re.sub(re_pattern, '', col) for col in law_nodong_rate_list]
law_nodong_won_list = [re.sub(re_pattern, '', col) for col in law_nodong_won_list]

law_nodong_rate['산업분류'] = law_nodong_rate_list
law_nodong_won['산업분류'] = law_nodong_won_list

law_nodong_won

,산업분류,연도,지표,비용
0,전체,2019,법정노동비용(계),381.7
1,광업(05~08),2019,법정노동비용(계),448.5
2,제조업(10~34),2019,법정노동비용(계),428.1
3,"전기, 가스, 증기 및 공기 조절 공급업(35)",2019,법정노동비용(계),605.5
4,"수도, 하수 및 폐기물 처리, 원료 재생업(36~39)",2019,법정노동비용(계),315.6
...,...,...,...,...
667,부동산업(68),2024,재해보상비,0.6
668,"전문, 과학 및 기술 서비스업(70~73)",2024,재해보상비,0
669,"사업시설 관리, 사업 지원 및 임대 서비스업(74~76)",2024,재해보상비,0.1
670,"예술, 스포츠 및 여가관련 서비스업(90~91)",2024,재해보상비,0


In [26]:
no_braket = r'\([0-9]+~?[0-9]+\)'

# 뒤에 숫자도 나가자
law_nodong_rate_list = [re.sub(no_braket, '', col) for col in law_nodong_rate_list]
law_nodong_won_list = [re.sub(no_braket, '', col) for col in law_nodong_won_list]

law_nodong_rate['산업분류'] = law_nodong_rate_list
law_nodong_won['산업분류'] = law_nodong_won_list

law_nodong_rate

,산업분류,연도,지표,수치
0,전체,2019,구성비(계),100
1,광업,2019,구성비(계),100
2,제조업,2019,구성비(계),100
3,"전기, 가스, 증기 및 공기 조절 공급업",2019,구성비(계),100
4,"수도, 하수 및 폐기물 처리, 원료 재생업",2019,구성비(계),100
...,...,...,...,...
667,부동산업,2024,재해보상비_(%),0.2
668,"전문, 과학 및 기술 서비스업",2024,재해보상비_(%),0
669,"사업시설 관리, 사업 지원 및 임대 서비스업",2024,재해보상비_(%),0
670,"예술, 스포츠 및 여가관련 서비스업",2024,재해보상비_(%),0


In [27]:
law_nodong_rate['지표'] = law_nodong_rate['지표'].str.replace('_(%)', '', regex=False)
law_nodong_mnu_rate['지표'] = law_nodong_mnu_rate['지표'].str.replace('_(%)', '', regex=False)
law_nodong_rate

,산업분류,연도,지표,수치
0,전체,2019,구성비(계),100
1,광업,2019,구성비(계),100
2,제조업,2019,구성비(계),100
3,"전기, 가스, 증기 및 공기 조절 공급업",2019,구성비(계),100
4,"수도, 하수 및 폐기물 처리, 원료 재생업",2019,구성비(계),100
...,...,...,...,...
667,부동산업,2024,재해보상비,0.2
668,"전문, 과학 및 기술 서비스업",2024,재해보상비,0
669,"사업시설 관리, 사업 지원 및 임대 서비스업",2024,재해보상비,0
670,"예술, 스포츠 및 여가관련 서비스업",2024,재해보상비,0


In [28]:
law_nodong_mnu_rate_list = [re.sub(re_pattern, '', col) for col in law_nodong_rate_list]
law_nodong_mnu_won_list = [re.sub(re_pattern, '', col) for col in law_nodong_won_list]

law_nodong_mnu_rate_list = [re.sub(no_braket, '', col) for col in law_nodong_rate_list]
law_nodong_mnu_won_list = [re.sub(no_braket, '', col) for col in law_nodong_won_list]

law_nodong_mnu_rate['산업분류'] = law_nodong_rate_list
law_nodong_mnu_won['산업분류'] = law_nodong_won_list

law_nodong_mnu_rate

,산업분류,연도,지표,수치
0,전체,2019,구성비(계),100
1,광업,2019,구성비(계),100
2,제조업,2019,구성비(계),100
3,"전기, 가스, 증기 및 공기 조절 공급업",2019,구성비(계),100
4,"수도, 하수 및 폐기물 처리, 원료 재생업",2019,구성비(계),100
...,...,...,...,...
667,부동산업,2024,재해보상비,0.2
668,"전문, 과학 및 기술 서비스업",2024,재해보상비,0
669,"사업시설 관리, 사업 지원 및 임대 서비스업",2024,재해보상비,0
670,"예술, 스포츠 및 여가관련 서비스업",2024,재해보상비,0


## 돈단위 변경 (천원->만원)

In [29]:
# 아놔... 얼탱이없네... 이게 왜 숫자냐고...
law_nodong_won['비용'] = law_nodong_won['비용'].astype(float)
law_nodong_mnu_won['비용'] = law_nodong_mnu_won['비용'].astype(float)

In [30]:
law_nodong_won['비용(만원)'] = law_nodong_won['비용'] / 10
law_nodong_mnu_won['비용(만원)'] = law_nodong_mnu_won['비용'] / 10

## 저장_최종.csv

In [31]:
law_nodong_mnu_rate.to_csv('data/law_nodong_rate.csv', index=False)
law_nodong_mnu_won.to_csv('data/law_nodong_won.csv', index=False)
law_nodong_rate.to_csv('data/law_nodong_total_rate.csv', index=False)
law_nodong_won.to_csv('data/law_nodong_total_won.csv', index=False)

# notlaw_nodong
## 헤더 나갓! 

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
notlaw_nodong.columns = notlaw_nodong.iloc[0]
notlaw_nodong_total.columns = notlaw_nodong_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
notlaw_nodong= notlaw_nodong.drop(0)
notlaw_nodong_total = notlaw_nodong_total.drop(0)

# 너도 나가!
notlaw_nodong.drop('기업규모별', axis=1, inplace=True)
notlaw_nodong_total.drop('기업규모별', axis=1, inplace=True)

In [ ]:
notlaw_nodong

In [ ]:
notlaw_nodong_total

In [ ]:
current_cols = notlaw_nodong_total.columns

## 괄호 나갓!!

In [ ]:
pattern = r' \(천원\)' # 두유노정규식
current_cols = [re.sub(pattern, '', col) for col in current_cols] # (천원은 다 빼주시고)
current_cols = [re.sub(r' \(%\)', '', col) for col in current_cols] # 퍼센트도 퇴근합니당

current_cols

## 연도 분리 및 멜트

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['법정외 복지비용(계)','주거비용','건강·보건비용','식사비용','교통·통신지원비용','보육지원금','보험료지원금','자녀학비보조비용','휴양·문화·체육·오락비용','우리사주제도 지원금','사내근로복지기금 출연금','기타']

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(notlaw_nodong.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
notlaw_nodong.columns = final_cols
notlaw_nodong_total.columns = final_cols

In [ ]:
notlaw_nodong_melted = notlaw_nodong.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')
notlaw_nodong_total_melted = notlaw_nodong_total.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')

notlaw_nodong_melted[['연도', '항목']] = notlaw_nodong_melted['항목'].str.split('_', expand=True, n=1)
notlaw_nodong_total_melted[['연도', '항목']] = notlaw_nodong_total_melted['항목'].str.split('_', expand=True, n=1)

notlaw_nodong_melted['비용'] = pd.to_numeric(notlaw_nodong_melted['비용'], errors='coerce')
notlaw_nodong_total_melted['비용'] = pd.to_numeric(notlaw_nodong_total_melted['비용'], errors='coerce')

notlaw_nodong_melted = notlaw_nodong_melted[['산업분류', '연도', '항목', '비용']]
notlaw_nodong_total_melted = notlaw_nodong_total_melted[['산업분류', '연도', '항목', '비용']]

In [ ]:
notlaw_nodong_melted

In [ ]:
notlaw_nodong_total_melted

## 알파벳 떼고 가실게요!

In [ ]:
notlaw_nodong_total_list = notlaw_nodong_melted['산업분류'].tolist()
notlaw_nodong_total_list

In [ ]:
notlaw_nodong_total_list = [re.sub(re_pattern, '', col) for col in notlaw_nodong_total_list]

notlaw_nodong_melted['산업분류'] = notlaw_nodong_total_list

notlaw_nodong_melted

In [ ]:
# 뒤에 숫자도 나가자
notlaw_nodong_total_list = [re.sub(no_braket, '', col) for col in notlaw_nodong_total_list]

notlaw_nodong_melted['산업분류'] = notlaw_nodong_total_list

notlaw_nodong_melted

## 돈단위 변경 (천원->만원)

In [ ]:
notlaw_nodong_melted['비용(만원)'] = notlaw_nodong_melted['비용'] / 10
notlaw_nodong_total_melted['비용(만원)'] = notlaw_nodong_total_melted['비용'] / 10

## 저장_최종.csv

In [ ]:
notlaw_nodong_melted.to_csv('data/notlaw_nodong_total.csv', index=False)
notlaw_nodong_total_melted.to_csv('data/notlaw_nodong.csv', index=False)
# 아놔 이름을 바꿔서 저장했어요... ㅡㅡ

- 아놔 둘이 이름을 바꿔서 저장했네요.. 내용이 뭔가 이상하더라니... ㅡㅡ 

# 산업재해 관련 처리

## 내용 좀 보고 가겠습니다.

In [ ]:
accident

In [ ]:
accident_total

- 이것도 손 많이 가겠는데...?

## 이제는 헤더가 퇴근해야 할 시간

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
accident.columns = accident.iloc[0]
accident_total.columns = accident_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
accident = accident.drop(0)
accident_total = accident_total.drop(0)

# 너도 나가!
accident.drop('업종별 중분류(1)', axis=1, inplace=True) # 얘 중분류 날립니다 (어차피 제조업 하나임)

In [ ]:
accident

In [ ]:
accident_total

## melt하기 전 처리
1. 연도가 2019~2024라서 그거 따로 붙여줄거고요
2. 우리 저 칼럼 보시면 (1) 있죠? 그거 떼버릴겁니다.
### 괄호는 안녕~

In [ ]:
# (1), (2) 떼고
accident.columns = accident.columns.str.strip()
accident_total.columns = accident_total.columns.str.strip()

accident.rename(columns={'업종별 중분류(2)':'업종별 중분류'}, inplace=True)
accident_total.rename(columns={'업종별 중분류(1)':'업종별 중분류'}, inplace=True)

In [ ]:
accident

In [ ]:
accident_total.columns

### 연도별로 구별해줄거에요

In [ ]:
# 연도향을 첨가해보아요
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['사업장수 (개소)', '근로자수 (명)', '재해자수 (명)', '사망자수 (명)', '재해율 (%)',
       '사망만인율 (‱)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(accident.columns) == len(final_cols) + 1:
    final_cols = ['업종별 중분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
accident.columns = final_cols
accident_total.columns = final_cols

In [ ]:
# 굿.
accident_total

In [ ]:
accident.columns

## 멜트로 재구성
- 이거 단위가 섞여있어서 하나를 둘로 찢을거예요. 하나는 사업장 수 대비 근로자수, 재해자수, 사망자수가 들어가고 다른 하나는 사업장 수 대비 재해율, 사망만인율(...퍼밀인가?)이 들어갈겁니다.

In [ ]:
# union_column = ['업종별 중분류']
# 그룹 1: 규모 및 수량 (사업장수, 근로자수, 재해자수, 사망자수)
qty_cols = [col for col in accident.columns if any(x in col for x in ['사업장수', '근로자수', '재해자수', '사망자수'])]
accident_quantity = accident.melt(id_vars=union_column, value_vars=qty_cols, var_name='항목', value_name='수치')

# 그룹 2: 비율 및 위험도 (재해율, 사망만인율)
rate_cols = [col for col in accident.columns if any(x in col for x in ['재해율', '사망만인율'])]
accident_rate = accident.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [accident_quantity, accident_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
union_column = ['업종별 중분류']

# 그룹 1: 규모 및 수량 (사업장수, 근로자수, 재해자수, 사망자수)
qty_cols = [col for col in accident.columns if any(x in col for x in ['사업장수', '근로자수', '재해자수', '사망자수'])]
accident_quantity = accident.melt(id_vars=union_column, value_vars=qty_cols, var_name='항목', value_name='수치')

# 그룹 2: 비율 및 위험도 (재해율, 사망만인율)
rate_cols = [col for col in accident.columns if any(x in col for x in ['재해율', '사망만인율'])]
accident_rate = accident.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [accident_quantity, accident_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
union_column = ['업종별 중분류']

# 그룹 1: 규모 및 수량 (사업장수, 근로자수, 재해자수, 사망자수)
qty_cols = [col for col in accident_total.columns if any(x in col for x in ['사업장수', '근로자수', '재해자수', '사망자수'])]
accident_total_quantity = accident_total.melt(id_vars=union_column, value_vars=qty_cols, var_name='항목', value_name='수치')

# 그룹 2: 비율 및 위험도 (재해율, 사망만인율)
rate_cols = [col for col in accident_total.columns if any(x in col for x in ['재해율', '사망만인율'])]
accident_total_rate = accident_total.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [accident_total_quantity, accident_total_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
accident_quantity = accident_quantity[['업종별 중분류','연도','지표','수치']]
accident_total_quantity = accident_total_quantity[['업종별 중분류','연도','지표','수치']]
accident_rate = accident_rate[['업종별 중분류','연도','지표','수치']]
accident_total_rate = accident_total_rate[['업종별 중분류','연도','지표','수치']]

## 저장_최종.csv

In [ ]:
accident_quantity.to_csv('data/accident_quantity.csv', index=False)
accident_rate.to_csv('data/accident_rate.csv', index=False)
accident_total_quantity.to_csv('data/accident_total_quantity.csv', index=False)
accident_total_rate.to_csv('data/accident_total_rate.csv', index=False)

# 산업 규모 및 임금별 근로시간

In [ ]:
payment_time

In [3]:
payment_time_total

,산업분류(1),규모별(1),2020,2020.1,2020.2,2020.3,2020.4,2020.5,2020.6,2020.7,...,2024.4,2024.5,2024.6,2024.7,2024.8,2024.9,2024.10,2024.11,2024.12,2024.13
0,산업분류(1),규모별(1),전체근로일수 (일),상용근로일수 (일),임시일용근로일수 (일),전체근로시간 (시간),상용총근로시간 (시간),상용소정실근로시간 (시간),상용초과근로시간 (시간),임시일용근로시간 (시간),...,상용총근로시간 (시간),상용소정실근로시간 (시간),상용초과근로시간 (시간),임시일용근로시간 (시간),전체임금총액 (원),상용임금총액 (원),상용정액급여 (원),상용초과급여 (원),상용특별급여 (원),임시일용임금총액 (원)
1,전체,전규모(1인이상),19.7,20.4,13.2,160.6,166.9,158.6,8.3,97.6,...,162.7,154.6,8.1,86,4079258,4337690,3555392,240059,542239,1808743
2,B.광업(05~08),전규모(1인이상),21.6,21.9,11.4,181.2,183.7,164.1,19.6,87.5,...,175.4,158.8,16.6,67.4,4899797,5080889,4161972,366480,552437,1570269
3,C.제조업(10~34),전규모(1인이상),20.2,20.4,14.1,172.7,174.5,157.6,16.9,113.9,...,172.1,155.2,16.9,100,4754798,4839081,3632918,468922,737241,2284327
4,"D.전기, 가스, 증기 및 공기 조절 공급업(35)",전규모(1인이상),19.3,19.4,18.6,163.7,164,154,10,148.3,...,156.8,148.3,8.6,142.4,7687480,7772397,5226552,381049,2164796,1888058
5,"E.수도, 하수 및 폐기물 처리, 원료 재생업(36~39)",전규모(1인이상),21.3,21.6,14.8,176.5,179.3,167.6,11.8,110.1,...,172.8,160.2,12.7,99,4430832,4569125,3649376,481372,438376,1797497
6,F.건설업(41~42),전규모(1인이상),16.9,20.7,12.7,136.9,169.4,163.7,5.7,101.7,...,164.4,159.5,4.9,90.7,3455292,4388217,3942894,175363,269960,2513485
7,G.도매 및 소매업(45~47),전규모(1인이상),20.5,20.9,13.4,163.8,167.7,162.6,5.1,93.6,...,162.5,158.1,4.4,93.3,4030938,4221809,3671773,118597,431439,1392416
8,H.운수 및 창고업(49~52),전규모(1인이상),19.9,20,16.1,159,159.9,150.9,9,114.1,...,162.5,151.8,10.7,81.9,4325511,4438141,3460442,355973,621726,1399140
9,I.숙박 및 음식점업(55~56),전규모(1인이상),18.9,21.7,11.7,149.7,177.2,172.1,5.1,78.1,...,168.8,162.2,6.6,62.7,2134030,2707329,2480445,137714,89170,819926


## 일단 헤더 들어내고...

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
payment_time.columns = payment_time.iloc[0]
payment_time_total.columns = payment_time_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
payment_time = payment_time.drop(0)
payment_time_total = payment_time_total.drop(0)

# 너도 나가!
payment_time.drop('규모별(1)', axis=1, inplace=True)
payment_time_total.drop('규모별(1)', axis=1, inplace=True)

In [ ]:
payment_time

In [ ]:
payment_time_total

## (1), (2)가 사라지는 마법!

In [ ]:
# (1), (2) 떼고
payment_time.columns = payment_time.columns.str.strip()
payment_time_total.columns = payment_time_total.columns.str.strip()

payment_time.rename(columns={'산업분류(2)':'산업분류'}, inplace=True)
payment_time_total.rename(columns={'산업분류(1)':'산업분류'}, inplace=True)

In [ ]:
payment_time.drop(columns='산업분류(1)', inplace=True) # 너도 나가

In [ ]:
payment_time

## 연도를 끼얹어주세용

In [ ]:
payment_time.columns

In [ ]:
# 연도향을 첨가해보아요
years = [ 2020, 2021, 2022, 2023, 2024]
base_names = ['전체근로일수 (일)', '상용근로일수 (일)', '임시일용근로일수 (일)', '전체근로시간 (시간)',
       '상용총근로시간 (시간)', '상용소정실근로시간 (시간)', '상용초과근로시간 (시간)', '임시일용근로시간 (시간)',
       '전체임금총액 (원)', '상용임금총액 (원)', '상용정액급여 (원)', '상용초과급여 (원)', '상용특별급여 (원)',
       '임시일용임금총액 (원)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(payment_time.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
payment_time.columns = final_cols
payment_time_total.columns = final_cols

In [ ]:
payment_time

## 쓰읍 얘도 분리해야것소...
- 일/시간 함께 묶고 원끼리 묶겠습니다.

In [ ]:
union_column = ['산업분류']

# 그룹 1: 일, 시간
day_cols = [col for col in payment_time.columns if any(x in col for x in ['(일)','(시간)'])]
payment_time_date = payment_time.melt(id_vars=union_column, value_vars=day_cols, var_name='항목', value_name='수치')

# 그룹 2: 돈
money_cols = [col for col in payment_time.columns if any(x in col for x in ['(원)'])]
payment_time_money = payment_time.melt(id_vars=union_column, value_vars=money_cols, var_name='항목', value_name='비용')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [payment_time_date, payment_time_money]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
payment_time_date = payment_time_date[['산업분류', '연도', '지표', '수치']]
payment_time_money = payment_time_money[['산업분류', '연도', '지표', '비용']]
payment_time_money

In [ ]:
# 똑같은거 한번 더 해주시면 됩니다.
union_column = ['산업분류']

# 그룹 1: 일, 시간
day_cols = [col for col in payment_time_total.columns if any(x in col for x in ['(일)','(시간)'])]
payment_time_total_date = payment_time_total.melt(id_vars=union_column, value_vars=day_cols, var_name='항목', value_name='수치')

# 그룹 2: 돈
money_cols = [col for col in payment_time_total.columns if any(x in col for x in ['(원)'])]
payment_time_total_money = payment_time_total.melt(id_vars=union_column, value_vars=money_cols, var_name='항목', value_name='비용')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [payment_time_total_date, payment_time_total_money]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
payment_time_total_date = payment_time_total_date[['산업분류', '연도', '지표', '수치']]
payment_time_total_money = payment_time_total_money[['산업분류', '연도', '지표', '비용']]

## 알파벳 나가!

In [ ]:
payment_total_list = payment_time_total_money['산업분류'].tolist()
payment_total_list

In [ ]:
payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_total_money['산업분류'] = payment_total_list

payment_time_total_money

In [ ]:
payment_total_list

In [ ]:
payment_total_list = payment_time_total_date['산업분류'].tolist()

payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_total_date['산업분류'] = payment_total_list

payment_time_total_date

In [ ]:
payment_total_list = payment_time_date['산업분류'].tolist()

payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_date['산업분류'] = payment_total_list

payment_time_date

In [ ]:
payment_total_list = payment_time_money['산업분류'].tolist()

payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_money['산업분류'] = payment_total_list

payment_time_money

## 어디가 돈단위 바꿔야지
- 얘는 원입니다. 네.
- 근데 저거 썡으로 나누면 소수점 아레 네자기라 보기 싫잖아요? 그래서 반올림할겁니다... ~~구레나룻... 아니... 소수점 아래 두자리는 남겨주세요~~

In [ ]:
payment_time_money['비용'] = payment_time_money['비용'].astype(float)
payment_time_total_money['비용'] = payment_time_total_money['비용'].astype(float)

In [ ]:
payment_time_money['비용(만원)'] = round(payment_time_money['비용'] / 10000, 2)
payment_time_total_money['비용(만원)'] = round(payment_time_total_money['비용'] / 10000, 2)

## 내 하드에 저-장

In [ ]:
payment_time_date.to_csv('data/payment_time_date.csv', index=False)
payment_time_money.to_csv('payment_time_money.csv', index=False)
payment_time_total_date.to_csv('data/payment_time_total_date.csv', index=False)
payment_time_total_money.to_csv('data/payment_time_total_money.csv', index=False)

## 추가 가공: 위 니드 어 콩캣
- 이게 산업 분류따라서 나뉘었습니다... 네...
- 근데 이걸 합쳐도 되나? 싶으시죠? 어차피 제조업 전체 볼거라서 합쳤습니다.

### 예들아 이제 나가줄래?

In [28]:
# 응 너 나가
payment_time_part1.drop('규모별(1)', axis=1, inplace=True)
payment_time_part2.drop('규모별(1)', axis=1, inplace=True)

In [29]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
payment_time_part1.columns = payment_time_part1.iloc[0]
payment_time_part2.columns = payment_time_part2.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
payment_time_part1 = payment_time_part1.drop(0)
payment_time_part2 = payment_time_part2.drop(0)

In [30]:
payment_time_part1.drop('산업분류(1)', axis = 1, inplace=True)

### 응 괄호도 나가

In [31]:
# (1), (2) 떼고
payment_time_part1.columns = payment_time_part1.columns.str.strip()
payment_time_part2.columns = payment_time_part2.columns.str.strip()

payment_time_part2.rename(columns={'산업분류별(1)':'산업분류'}, inplace=True)

In [32]:
payment_time_part2

,산업분류,전체근로일수 (일),상용근로일수 (일),임시일용근로일수 (일),전체근로시간 (시간),상용총근로시간 (시간),상용소정실근로시간 (시간),상용초과근로시간 (시간),임시일용근로시간 (시간),전체임금총액 (원),...,상용총근로시간 (시간),상용소정실근로시간 (시간),상용초과근로시간 (시간),임시일용근로시간 (시간),전체임금총액 (원),상용임금총액 (원),상용정액급여 (원),상용초과급여 (원),상용특별급여 (원),임시일용임금총액 (원)
1,C. 제조업(10~33),21.3,21.4,17.4,184.9,186.7,163.3,23.3,135.8,3462028,...,177.9,158.9,19,116.2,4017254,4094454,3006878,396550,691025,1656305


### 연도 추가 후 칼람명 변경

In [33]:
# 연도향을 첨가해보아요
years = [ 2020, 2021, 2022, 2023, 2024]
base_names = ['전체근로일수 (일)', '상용근로일수 (일)', '임시일용근로일수 (일)', '전체근로시간 (시간)',
       '상용총근로시간 (시간)', '상용소정실근로시간 (시간)', '상용초과근로시간 (시간)', '임시일용근로시간 (시간)',
       '전체임금총액 (원)', '상용임금총액 (원)', '상용정액급여 (원)', '상용초과급여 (원)', '상용특별급여 (원)',
       '임시일용임금총액 (원)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(payment_time.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
payment_time_part1.columns = final_cols

In [17]:
payment_time_part1

,2020_전체근로일수 (일),2020_상용근로일수 (일),2020_임시일용근로일수 (일),2020_전체근로시간 (시간),2020_상용총근로시간 (시간),2020_상용소정실근로시간 (시간),2020_상용초과근로시간 (시간),2020_임시일용근로시간 (시간),2020_전체임금총액 (원),2020_상용임금총액 (원),...,2024_상용총근로시간 (시간),2024_상용소정실근로시간 (시간),2024_상용초과근로시간 (시간),2024_임시일용근로시간 (시간),2024_전체임금총액 (원),2024_상용임금총액 (원),2024_상용정액급여 (원),2024_상용초과급여 (원),2024_상용특별급여 (원),2024_임시일용임금총액 (원)
0,전체근로일수 (일),상용근로일수 (일),임시일용근로일수 (일),전체근로시간 (시간),상용총근로시간 (시간),상용소정실근로시간 (시간),상용초과근로시간 (시간),임시일용근로시간 (시간),전체임금총액 (원),상용임금총액 (원),...,상용총근로시간 (시간),상용소정실근로시간 (시간),상용초과근로시간 (시간),임시일용근로시간 (시간),전체임금총액 (원),상용임금총액 (원),상용정액급여 (원),상용초과급여 (원),상용특별급여 (원),임시일용임금총액 (원)
1,20.2,20.4,14.1,172.7,174.5,157.6,16.9,113.9,3990086,4058423,...,172.1,155.2,16.9,100,4754798,4839081,3632918,468922,737241,2284327


In [34]:
# 연도향을 첨가해보아요
years = [2015, 2016, 2017, 2018, 2019]
base_names = ['전체근로일수 (일)', '상용근로일수 (일)', '임시일용근로일수 (일)', '전체근로시간 (시간)',
       '상용총근로시간 (시간)', '상용소정실근로시간 (시간)', '상용초과근로시간 (시간)', '임시일용근로시간 (시간)',
       '전체임금총액 (원)', '상용임금총액 (원)', '상용정액급여 (원)', '상용초과급여 (원)', '상용특별급여 (원)',
       '임시일용임금총액 (원)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(payment_time_part2.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
payment_time_part2.columns = final_cols

In [35]:
payment_time_part2

,산업분류,2015_전체근로일수 (일),2015_상용근로일수 (일),2015_임시일용근로일수 (일),2015_전체근로시간 (시간),2015_상용총근로시간 (시간),2015_상용소정실근로시간 (시간),2015_상용초과근로시간 (시간),2015_임시일용근로시간 (시간),2015_전체임금총액 (원),...,2019_상용총근로시간 (시간),2019_상용소정실근로시간 (시간),2019_상용초과근로시간 (시간),2019_임시일용근로시간 (시간),2019_전체임금총액 (원),2019_상용임금총액 (원),2019_상용정액급여 (원),2019_상용초과급여 (원),2019_상용특별급여 (원),2019_임시일용임금총액 (원)
1,C. 제조업(10~33),21.3,21.4,17.4,184.9,186.7,163.3,23.3,135.8,3462028,...,177.9,158.9,19,116.2,4017254,4094454,3006878,396550,691025,1656305


In [39]:
# '제조업' 단어가 포함된 모든 셀을 '제조업'으로 변경
payment_time_part2.loc[payment_time_part2['산업분류'].str.contains('제조업', na=False), '산업분류'] = '제조업'

In [40]:
payment_time_part2

,산업분류,2015_전체근로일수 (일),2015_상용근로일수 (일),2015_임시일용근로일수 (일),2015_전체근로시간 (시간),2015_상용총근로시간 (시간),2015_상용소정실근로시간 (시간),2015_상용초과근로시간 (시간),2015_임시일용근로시간 (시간),2015_전체임금총액 (원),...,2019_상용총근로시간 (시간),2019_상용소정실근로시간 (시간),2019_상용초과근로시간 (시간),2019_임시일용근로시간 (시간),2019_전체임금총액 (원),2019_상용임금총액 (원),2019_상용정액급여 (원),2019_상용초과급여 (원),2019_상용특별급여 (원),2019_임시일용임금총액 (원)
1,제조업,21.3,21.4,17.4,184.9,186.7,163.3,23.3,135.8,3462028,...,177.9,158.9,19,116.2,4017254,4094454,3006878,396550,691025,1656305


### 묶자...

In [49]:
payment_time_all = pd.concat([payment_time_part2, payment_time_part1], axis=1)
payment_time_all

,산업분류,2015_전체근로일수 (일),2015_상용근로일수 (일),2015_임시일용근로일수 (일),2015_전체근로시간 (시간),2015_상용총근로시간 (시간),2015_상용소정실근로시간 (시간),2015_상용초과근로시간 (시간),2015_임시일용근로시간 (시간),2015_전체임금총액 (원),...,2024_상용총근로시간 (시간),2024_상용소정실근로시간 (시간),2024_상용초과근로시간 (시간),2024_임시일용근로시간 (시간),2024_전체임금총액 (원),2024_상용임금총액 (원),2024_상용정액급여 (원),2024_상용초과급여 (원),2024_상용특별급여 (원),2024_임시일용임금총액 (원)
1,제조업,21.3,21.4,17.4,184.9,186.7,163.3,23.3,135.8,3462028,...,172.1,155.2,16.9,100,4754798,4839081,3632918,468922,737241,2284327


### 네 이제 멜트하고 단위 바꿔주시면 됩니다.

In [50]:
union_column = ['산업분류']

# 그룹 1: 일, 시간
day_cols = [col for col in payment_time_all.columns if any(x in col for x in ['(일)','(시간)'])]
payment_time_all_date = payment_time_all.melt(id_vars=union_column, value_vars=day_cols, var_name='항목', value_name='수치')

# 그룹 2: 돈
money_cols = [col for col in payment_time_all.columns if any(x in col for x in ['(원)'])]
payment_time_all_money = payment_time_all.melt(id_vars=union_column, value_vars=money_cols, var_name='항목', value_name='비용')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [payment_time_all_date, payment_time_all_money]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [51]:
payment_time_all_date

,산업분류,수치,연도,지표
0,제조업,21.3,2015,전체근로일수
1,제조업,21.4,2015,상용근로일수
2,제조업,17.4,2015,임시일용근로일수
3,제조업,184.9,2015,전체근로시간
4,제조업,186.7,2015,상용총근로시간
...,...,...,...,...
75,제조업,169.7,2024,전체근로시간
76,제조업,172.1,2024,상용총근로시간
77,제조업,155.2,2024,상용소정실근로시간
78,제조업,16.9,2024,상용초과근로시간


#### 왜 맨날 순서가 뻑나는것이며

In [52]:
payment_time_all_date = payment_time_all_date[['산업분류', '연도', '지표', '수치']]
payment_time_all_money = payment_time_all_money[['산업분류', '연도', '지표', '비용']]

In [53]:
payment_time_all_money

,산업분류,연도,지표,비용
0,제조업,2015,전체임금총액,3462028
1,제조업,2015,상용임금총액,3537204
2,제조업,2015,상용정액급여,2521507
3,제조업,2015,상용초과급여,378569
4,제조업,2015,상용특별급여,637128
5,제조업,2015,임시일용임금총액,1456963
6,제조업,2016,전체임금총액,3602539
7,제조업,2016,상용임금총액,3668846
8,제조업,2016,상용정액급여,2612797
9,제조업,2016,상용초과급여,389325


### 돈단위 (만원)으로 변경

In [54]:
payment_time_all_money['비용'] = payment_time_all_money['비용'].astype(float)
payment_time_all_money['비용(만원)'] = round(payment_time_all_money['비용'] / 10000, 2)

### 내 하드에 저장

In [56]:
payment_time_all_date.to_csv('data/payment_time_all_date.csv', index=False)
payment_time_all_money.to_csv('data/payment_time_all_money.csv', index=False)

# 손익계산서
- 저 사실 재무제표 어느정도는 볼 줄 압니다. 네.
- 행에는 안 쓰여있지만 단위가 (백만원)입니다. 

In [ ]:
sonic

In [ ]:
sonic_jp

In [ ]:
sonic_jejo

In [ ]:
sonic_jp_jejo

## 저기 기업규모 날리고 가실게요~

In [ ]:
# 날리고 멜팅합시다.
df_list = [sonic, sonic_jp, sonic_jejo, sonic_jp_jejo]

for df in df_list:
    df.drop('기업규모별', axis=1, inplace=True)

In [ ]:
sonic

- 이 얼탱이없는 반복문은 왜 나왔느냐... 간단합니다. 똑같은거 네 개 할거면 걍 반복문 돌려도 되지 않음? 해서 나온겁니다.

## 업종코드 제거

In [ ]:
sonic['업종코드별'].tolist()
# 알파벳이 한글자 아니면 세글자네요.

In [ ]:
no_code = r'[A-Z,0-9, -]+' # 규식정

for df in df_list:
    df_col = df['업종코드별'].tolist()
    df_col = [re.sub(no_code, '', col) for col in df_col]
    df['업종코드별'] = df_col # 작용_최종.py


In [ ]:
sonic

## 멜트다운!
- 근데 반복문 돌릴거라 저장도 같이 되는...

In [ ]:
# 파이참은 셀이 100개가 넘어가면 뻗는군요...
# 멜트다운을 어떻게 할거냐면 업종-년도-계정과목-액수로 할거예요. 감 좀 오시져?
# 그렇게 해야 우리가 그룹바이 하기가 편합니다. 대신 얘는 칼럼갖고 노가다는 안 해도 되니 다행이군요.
filename_list = ['data/sonic.csv', 'data/sonic_jp.csv', 'data/sonic_jejo.csv', 'data/sonic_jp_jejo.csv']
# jp = Job Paymemt의 약자(재팬 아님)

for i, df in enumerate(df_list):
    df_melted = df.melt(id_vars=['업종코드별', '계정항목별'], var_name='연도', value_name='비용')
    df_melted['비용'] = pd.to_numeric(df_melted['비용'], errors='coerce') # 인자 너는 숫자다잉
    df_melted['연도'] = df_melted['연도'].astype(int) # 응 너도
    df_melted.to_csv(filename_list[i], index=False)